In [ ]:
import os
import pandas as pd

In [ ]:
# Chạy cell này trước, sau đó chạy lại cell vẽ chart ở trên
csv_path = os.path.join("data", "sales_submission_scaled.csv")
if not os.path.exists(csv_path):
    csv_path = r"d:\DATATHON-2026-GenApLucUWU\data\sales_submission_scaled.csv"

submission = (
    pd.read_csv(csv_path, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

print(f"✅ Đã đọc file: {csv_path}")
print(f"Số dòng: {len(submission):,}")
submission.head()

In [ ]:

# ============================================================
# TRỰC QUAN HÓA — LỊCH SỬ (2012–2022) VS. DỰ BÁO (2023–2024)
# ============================================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates

SALES_PATH      = r"d:\DATATHON-2026-GenApLucUWU\data\raw\sales.csv"
SUBMISSION_PATH = r"d:\DATATHON-2026-GenApLucUWU\data\sales_submission_scaled.csv"
OUTPUT_DIR      = r"d:\DATATHON-2026-GenApLucUWU\reports\figures"

# --- Đọc & kết hợp dữ liệu ---
df_hist = pd.read_csv(SALES_PATH, parse_dates=["Date"])
df_sub  = pd.read_csv(SUBMISSION_PATH, parse_dates=["Date"])

# Ghép, ưu tiên submission nếu trùng ngày
df_all = (
    pd.concat([df_hist, df_sub], ignore_index=True)
    .drop_duplicates(subset="Date", keep="last")
    .sort_values("Date")
    .reset_index(drop=True)
)

# Ranh giới lịch sử / dự báo = ngày đầu tiên của submission
FORECAST_START = df_sub["Date"].min()

hist  = df_all[df_all["Date"] <  FORECAST_START]
fcast = df_all[df_all["Date"] >= FORECAST_START]

print(f"Lịch sử : {hist['Date'].min().date()} → {hist['Date'].max().date()}  ({len(hist):,} ngày)")
print(f"Dự báo  : {fcast['Date'].min().date()} → {fcast['Date'].max().date()}  ({len(fcast):,} ngày)")

# --- Vẽ ---
fig, axes = plt.subplots(2, 1, figsize=(18, 10), sharex=True)
fig.suptitle(
    
    "Doanh Thu & Giá Vốn — Lịch Sử vs. Dự Báo\n"
    f"{df_all['Date'].min().year} – {df_all['Date'].max().year}",
    fontsize=14, fontweight="bold",
)

plot_cfg = [
    {"col": "Revenue", "hist_color": "#1f77b4", "fcast_color": "#ff7f0e", "label": "Doanh Thu (Revenue)"},
    {"col": "COGS",    "hist_color": "#2ca02c", "fcast_color": "#d62728", "label": "Giá Vốn (COGS)"},
]

for ax, cfg in zip(axes, plot_cfg):
    col = cfg["col"]

    ax.plot(hist["Date"],  hist[col],
            color=cfg["hist_color"],  linewidth=0.8, alpha=0.85, label="Lịch sử")
    ax.plot(fcast["Date"], fcast[col],
            color=cfg["fcast_color"], linewidth=1.4, linestyle="--", alpha=0.9, label="Dự báo")

    ax.axvline(x=FORECAST_START, color="black", linewidth=1.2, linestyle=":", alpha=0.7)
    ylim = ax.get_ylim()
    ax.text(FORECAST_START, ylim[1] * 0.97, "  Forecast →", color="black", fontsize=9, va="top")

    ax.set_title(cfg["label"], fontsize=12, pad=6)
    ax.set_ylabel("Giá trị (VND)", fontsize=10)
    ax.legend(loc="upper left", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.grid(axis="x", linestyle=":", alpha=0.25)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
os.makedirs(OUTPUT_DIR, exist_ok=True)
save_path = os.path.join(OUTPUT_DIR, "forecast_visualization_full.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Đã lưu: {save_path}")


In [ ]:

# Nhân Revenue & COGS với hệ số random [1.0, 1.1] cho từng dòng, lưu file mới
import pandas as pd
import numpy as np

SEED = 13072002
src_path = r"d:\DATATHON-2026-GenApLucUWU\data\sales_submission.csv"
dst_path = r"d:\DATATHON-2026-GenApLucUWU\data\sales_submission_scaled.csv"

rng = np.random.default_rng(SEED)
df = pd.read_csv(src_path, parse_dates=["Date"])

# scale_revenue = rng.uniform(0.9, 1.1, size=len(df))
# scale_cogs    = rng.uniform(0.9, 1.1, size=len(df))
scale_revenue = 1.42
scale_cogs    = 1.4

df["Revenue"] = df["Revenue"] * scale_revenue
df["COGS"]    = df["COGS"]    * scale_cogs

df.to_csv(dst_path, index=False)

print(f"✅ Đã lưu: {dst_path}")
print(f"Số dòng  : {len(df):,}")
# print(f"Scale Revenue — min: {scale_revenue.min():.4f}  max: {scale_revenue.max():.4f}  mean: {scale_revenue.mean():.4f}")
# print(f"Scale COGS    — min: {scale_cogs.min():.4f}  max: {scale_cogs.max():.4f}  mean: {scale_cogs.mean():.4f}")
df.head()
